# Step Counter - CNN Model Development

This notebook trains a shallow CNN architecture for step counting.

## Approach
1. Load preprocessed data from `data_preparation_cnn.ipynb`
2. Import ShallowCNN model architecture from `src/models/`
3. Train model with binary classification
4. Evaluate performance on validation set
5. Save model for final test evaluation (in separate notebook)

## 1. Configuration and Imports

In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Import model architecture
from src.models.shallow_cnn import ShallowCNN
from src.models.deep_cnn import DeepCNN

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch version: 2.5.1
Device: cpu


In [2]:
# Configuration

# MODEL_TYPE: Choose which model to train ('shallow_cnn', 'deep_cnn', or 'wavenet')
MODEL_TYPE = 'deep_cnn'

CONFIG = {
    # Data
    'data_dir': Path('../data/processed'),
    'task_type': 'regression',  # 'binary' for step detection, 'regression' for step counting
    
    # Training hyperparameters
    'batch_size': 32,
    'epochs': 40,
    'learning_rate': 0.01,
    'patience': 10,  # Early stopping patience
    
    # Model hyperparameters
    'n_filters': 64,
    'dropout_rate': 0.3,
    
    # Output
    'save_dir': Path('../models/saved'),
    'experiment_name': f'{MODEL_TYPE}_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    
    # Optional: Add descriptive notes for this experiment
    'notes': 'increased lr, decreased dropout'
}

# Create experiment directory
experiment_dir = CONFIG['save_dir'] / CONFIG['experiment_name']
experiment_dir.mkdir(parents=True, exist_ok=True)
CONFIG['experiment_dir'] = experiment_dir

print('Configuration:')
print(json.dumps({k: str(v) for k, v in CONFIG.items()}, indent=2))

Configuration:
{
  "data_dir": "..\\data\\processed",
  "task_type": "regression",
  "batch_size": "32",
  "epochs": "40",
  "learning_rate": "0.01",
  "patience": "10",
  "n_filters": "64",
  "dropout_rate": "0.3",
  "save_dir": "..\\models\\saved",
  "experiment_name": "deep_cnn_20260311_150756",
  "notes": "increased lr, decreased dropout",
  "experiment_dir": "..\\models\\saved\\deep_cnn_20260311_150756"
}


## 2. Load Preprocessed Data

Load the train/val/test splits created in `data_preparation_cnn.ipynb`.

In [3]:
# Determine which label to load based on task_type
label_key = 'y_binary' if CONFIG['task_type'] == 'binary' else 'y_count'

# Load training data
train_data = np.load(CONFIG['data_dir'] / 'cnn_train_data.npz')
X_train = train_data['X']
y_train = train_data[label_key].astype(np.float32)

# Load validation data
val_data = np.load(CONFIG['data_dir'] / 'cnn_val_data.npz')
X_val = val_data['X']
y_val = val_data[label_key].astype(np.float32)

# Load test data
test_data = np.load(CONFIG['data_dir'] / 'cnn_test_data.npz')
X_test = test_data['X']
y_test = test_data[label_key].astype(np.float32)

# Convert to PyTorch tensors (PyTorch expects channels first: N, C, L)
X_train_tensor = torch.FloatTensor(X_train).permute(0, 2, 1)  # (N, L, C) -> (N, C, L)
y_train_tensor = torch.FloatTensor(y_train)
X_val_tensor = torch.FloatTensor(X_val).permute(0, 2, 1)
y_val_tensor = torch.FloatTensor(y_val)
X_test_tensor = torch.FloatTensor(X_test).permute(0, 2, 1)
y_test_tensor = torch.FloatTensor(y_test)

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print('Data Loaded:')
print(f'  X_train: {X_train.shape} -> Tensor: {X_train_tensor.shape}')
print(f'  y_train: {y_train.shape} -> Tensor: {y_train_tensor.shape}')
print(f'  X_val: {X_val.shape} -> Tensor: {X_val_tensor.shape}')
print(f'  y_val: {y_val.shape} -> Tensor: {y_val_tensor.shape}')
print(f'  X_test: {X_test.shape} -> Tensor: {X_test_tensor.shape}')
print(f'  y_test: {y_test.shape} -> Tensor: {y_test_tensor.shape}')
print(f'\nTask: {"Binary classification (has steps or not)" if CONFIG["task_type"] == "binary" else "Regression (step count prediction)"}')
print(f'Label: {label_key}')
print(f'Input shape per sample (channels, length): {X_train_tensor.shape[1:]}')
print(f'Batch size: {CONFIG["batch_size"]}')
print(f'Number of batches - Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

Data Loaded:
  X_train: (81272, 200, 4) -> Tensor: torch.Size([81272, 4, 200])
  y_train: (81272,) -> Tensor: torch.Size([81272])
  X_val: (28787, 200, 4) -> Tensor: torch.Size([28787, 4, 200])
  y_val: (28787,) -> Tensor: torch.Size([28787])
  X_test: (26267, 200, 4) -> Tensor: torch.Size([26267, 4, 200])
  y_test: (26267,) -> Tensor: torch.Size([26267])

Task: Regression (step count prediction)
Label: y_count
Input shape per sample (channels, length): torch.Size([4, 200])
Batch size: 32
Number of batches - Train: 2540, Val: 900, Test: 821


## 3. Training Utilities

Create reusable functions for training and evaluation.

In [4]:
def train_model(model, train_loader, val_loader, config):
    """
    Train a model with early stopping and learning rate scheduling.
    
    Args:
        model: PyTorch model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        config: Configuration dictionary
    
    Returns:
        history: Training history dictionary
        model: Trained model
    """
    
    model_name = config.get('model_type', 'Model').replace('_', ' ').title()
    
    print(f'\n{"="*60}')
    print(f'Training {model_name}')
    print(f'Experiment: {config["experiment_name"]}')
    print(f'{"="*60}')
    
    # Move model to device
    model = model.to(device)
    
    # Model summary
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\nTotal parameters: {total_params:,}')
    print(f'Trainable parameters: {trainable_params:,}')
    
    # Loss function based on task type
    if config['task_type'] == 'binary':
        criterion = nn.BCEWithLogitsLoss()
        print(f'\nTask: Binary classification')
        print(f'Loss function: BCEWithLogitsLoss')
    else:
        criterion = nn.MSELoss()
        print(f'\nTask: Regression (step counting)')
        print(f'Loss function: MSELoss')
    
    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6, verbose=True
    )
    
    # Training history
    if config['task_type'] == 'binary':
        history = {
            'loss': [],
            'val_loss': [],
            'accuracy': [],
            'val_accuracy': [],
            'precision': [],
            'val_precision': [],
            'recall': [],
            'val_recall': []
        }
    else:
        history = {
            'loss': [],
            'val_loss': [],
            'mae': [],
            'val_mae': [],
            'rmse': [],
            'val_rmse': []
        }
    
    # Early stopping variables
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    best_epoch = 0
    
    # Training loop
    for epoch in range(config['epochs']):
        # Training phase
        model.train()
        train_loss = 0.0
        train_preds = []
        train_targets = []
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X).squeeze()
            
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_X.size(0)
            train_preds.extend(outputs.detach().cpu().numpy())
            train_targets.extend(batch_y.cpu().numpy())
        
        train_loss /= len(train_loader.dataset)
        train_preds = np.array(train_preds)
        train_targets = np.array(train_targets)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                outputs = model(batch_X).squeeze()
                loss = criterion(outputs, batch_y)
                
                val_loss += loss.item() * batch_X.size(0)
                val_preds.extend(outputs.cpu().numpy())
                val_targets.extend(batch_y.cpu().numpy())
        
        val_loss /= len(val_loader.dataset)
        val_preds = np.array(val_preds)
        val_targets = np.array(val_targets)
        
        # Calculate metrics based on task type
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        if config['task_type'] == 'binary':
            # Apply sigmoid for binary classification (since we use BCEWithLogitsLoss)
            train_preds_prob = 1 / (1 + np.exp(-train_preds))
            val_preds_prob = 1 / (1 + np.exp(-val_preds))
            
            train_pred_binary = (train_preds_prob >= 0.5).astype(int)
            val_pred_binary = (val_preds_prob >= 0.5).astype(int)
            
            train_acc = (train_pred_binary == train_targets).mean()
            val_acc = (val_pred_binary == val_targets).mean()
            
            # Precision and recall
            train_tp = ((train_pred_binary == 1) & (train_targets == 1)).sum()
            train_fp = ((train_pred_binary == 1) & (train_targets == 0)).sum()
            train_fn = ((train_pred_binary == 0) & (train_targets == 1)).sum()
            train_precision = train_tp / (train_tp + train_fp) if (train_tp + train_fp) > 0 else 0
            train_recall = train_tp / (train_tp + train_fn) if (train_tp + train_fn) > 0 else 0
            
            val_tp = ((val_pred_binary == 1) & (val_targets == 1)).sum()
            val_fp = ((val_pred_binary == 1) & (val_targets == 0)).sum()
            val_fn = ((val_pred_binary == 0) & (val_targets == 1)).sum()
            val_precision = val_tp / (val_tp + val_fp) if (val_tp + val_fp) > 0 else 0
            val_recall = val_tp / (val_tp + val_fn) if (val_tp + val_fn) > 0 else 0
            
            history['accuracy'].append(train_acc)
            history['val_accuracy'].append(val_acc)
            history['precision'].append(train_precision)
            history['val_precision'].append(val_precision)
            history['recall'].append(train_recall)
            history['val_recall'].append(val_recall)
            
            print(f'Epoch {epoch+1}/{config["epochs"]} - '
                  f'loss: {train_loss:.4f} - acc: {train_acc:.4f} - '
                  f'val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')
        else:
            # Regression metrics
            train_mae = np.abs(train_preds - train_targets).mean()
            val_mae = np.abs(val_preds - val_targets).mean()
            train_rmse = np.sqrt(((train_preds - train_targets) ** 2).mean())
            val_rmse = np.sqrt(((val_preds - val_targets) ** 2).mean())
            
            history['mae'].append(train_mae)
            history['val_mae'].append(val_mae)
            history['rmse'].append(train_rmse)
            history['val_rmse'].append(val_rmse)
            
            print(f'Epoch {epoch+1}/{config["epochs"]} - '
                  f'loss: {train_loss:.4f} - mae: {train_mae:.4f} - '
                  f'val_loss: {val_loss:.4f} - val_mae: {val_mae:.4f}')
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            
            # Save best checkpoint in experiment directory
            checkpoint_path = config['experiment_dir'] / 'best_model.pth'
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'config': config
            }, checkpoint_path)
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                print(f'\nEarly stopping triggered after {epoch+1} epochs')
                print(f'Best model was at epoch {best_epoch}')
                break
    
    # Restore best weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    # Add training metadata to history
    history['best_epoch'] = best_epoch
    history['total_epochs'] = epoch + 1
    
    print(f'\nTraining complete!')
    print(f'Best epoch: {best_epoch}')
    print(f'Best validation loss: {best_val_loss:.4f}')
    
    return history, model


print('Training function defined')

Training function defined


In [5]:
def plot_training_history(history):
    """
    Plot training and validation metrics.
    
    Args:
        history: Training history dictionary
    """
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0, 0].plot(history['loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Check if binary classification or regression
    if 'accuracy' in history:
        # Binary classification metrics
        # Accuracy
        axes[0, 1].plot(history['accuracy'], label='Train Accuracy', linewidth=2)
        axes[0, 1].plot(history['val_accuracy'], label='Val Accuracy', linewidth=2)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].set_title('Accuracy')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Precision
        axes[1, 0].plot(history['precision'], label='Train Precision', linewidth=2)
        axes[1, 0].plot(history['val_precision'], label='Val Precision', linewidth=2)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Precision')
        axes[1, 0].set_title('Precision')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Recall
        axes[1, 1].plot(history['recall'], label='Train Recall', linewidth=2)
        axes[1, 1].plot(history['val_recall'], label='Val Recall', linewidth=2)
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Recall')
        axes[1, 1].set_title('Recall')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    else:
        # Regression metrics
        # MAE
        axes[0, 1].plot(history['mae'], label='Train MAE', linewidth=2)
        axes[0, 1].plot(history['val_mae'], label='Val MAE', linewidth=2)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('MAE')
        axes[0, 1].set_title('Mean Absolute Error')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # RMSE
        axes[1, 0].plot(history['rmse'], label='Train RMSE', linewidth=2)
        axes[1, 0].plot(history['val_rmse'], label='Val RMSE', linewidth=2)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('RMSE')
        axes[1, 0].set_title('Root Mean Squared Error')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # Empty plot (or could add another metric)
        axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()


print('Visualization function defined')

Visualization function defined


## 4. Train ShallowCNN Model

Train the shallow CNN architecture with binary classification.

In [6]:
# Import WaveNet if needed
from src.models.wavenet import WaveNet

# Create model based on MODEL_TYPE
if MODEL_TYPE == 'shallow_cnn':
    model = ShallowCNN(
        input_channels=X_train.shape[2],
        sequence_length=X_train.shape[1],
        num_filters=CONFIG['n_filters'],
        output_type=CONFIG['task_type'],
        dropout_rate=CONFIG['dropout_rate']
    )
elif MODEL_TYPE == 'deep_cnn':
    model = DeepCNN(
        input_channels=X_train.shape[2],
        sequence_length=X_train.shape[1],
        num_filters=CONFIG['n_filters'],
        output_type=CONFIG['task_type'],
        dropout_rate=CONFIG['dropout_rate']
    )
elif MODEL_TYPE == 'wavenet':
    model = WaveNet(
        input_channels=X_train.shape[2],
        sequence_length=X_train.shape[1],
        num_filters=CONFIG['n_filters'],
        output_type=CONFIG['task_type'],
        dropout_rate=CONFIG['dropout_rate']
    )
else:
    raise ValueError(f"Unknown model type: {MODEL_TYPE}. Choose 'shallow_cnn', 'deep_cnn', or 'wavenet'")

print(f'Model created: {MODEL_TYPE}')
print(f'Input shape: (window_size={X_train.shape[1]}, n_channels={X_train.shape[2]})')
print(f'Output type: {CONFIG["task_type"]}')
print(f'Filters: {CONFIG["n_filters"]}, Dropout: {CONFIG["dropout_rate"]}')

Model created: deep_cnn
Input shape: (window_size=200, n_channels=4)
Output type: regression
Filters: 64, Dropout: 0.3


### Train Model

In [ ]:
# Train the model
history, model = train_model(model, train_loader, val_loader, CONFIG)

# Plot training history
plot_training_history(history)


Training Model
Experiment: deep_cnn_20260311_150756

Total parameters: 1,563,201
Trainable parameters: 1,563,201

Task: Regression (step counting)
Loss function: MSELoss


C:\Users\PekkoKauppila\anaconda3\envs\ml-projects\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/40 - loss: 0.4762 - mae: 0.4140 - val_loss: 0.3370 - val_mae: 0.3387
Epoch 2/40 - loss: 0.4024 - mae: 0.3720 - val_loss: 0.4740 - val_mae: 0.4503
Epoch 3/40 - loss: 0.3995 - mae: 0.3580 - val_loss: 0.3146 - val_mae: 0.3922
Epoch 4/40 - loss: 0.4110 - mae: 0.3521 - val_loss: 0.3519 - val_mae: 0.3481
Epoch 5/40 - loss: 0.4282 - mae: 0.3544 - val_loss: 0.3293 - val_mae: 0.3305
Epoch 6/40 - loss: 0.4206 - mae: 0.3474 - val_loss: 0.2948 - val_mae: 0.3347
Epoch 7/40 - loss: 0.4724 - mae: 0.3796 - val_loss: 0.3275 - val_mae: 0.3545


## 5. Evaluate on Validation Set

Evaluate the trained model on the validation set. Use this for hyperparameter tuning and model selection.

In [ ]:
# Move model to device and set to eval mode
model = model.to(device)
model.eval()

# Make predictions on validation set
val_preds = []
val_targets = []

with torch.no_grad():
    for batch_X, batch_y in val_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X).squeeze()
        
        val_preds.extend(outputs.cpu().numpy())
        val_targets.extend(batch_y.numpy())

val_preds = np.array(val_preds)
val_targets = np.array(val_targets)

print(f'\n{"="*60}')
print(f'VALIDATION SET RESULTS')
print(f'{"="*60}')

if CONFIG['task_type'] == 'binary':
    # Binary classification metrics
    # Apply sigmoid to logits
    val_preds_prob = 1 / (1 + np.exp(-val_preds))
    val_pred_binary = (val_preds_prob >= 0.5).astype(int)
    
    # Calculate metrics
    val_acc = (val_pred_binary == val_targets).mean()
    
    val_tp = ((val_pred_binary == 1) & (val_targets == 1)).sum()
    val_fp = ((val_pred_binary == 1) & (val_targets == 0)).sum()
    val_fn = ((val_pred_binary == 0) & (val_targets == 1)).sum()
    val_tn = ((val_pred_binary == 0) & (val_targets == 0)).sum()
    
    val_precision = val_tp / (val_tp + val_fp) if (val_tp + val_fp) > 0 else 0
    val_recall = val_tp / (val_tp + val_fn) if (val_tp + val_fn) > 0 else 0
    val_f1 = 2 * (val_precision * val_recall) / (val_precision + val_recall) if (val_precision + val_recall) > 0 else 0
    
    # Calculate loss
    criterion = nn.BCEWithLogitsLoss()
    val_loss = criterion(torch.FloatTensor(val_preds), torch.FloatTensor(val_targets)).item()
    
    print(f'\nMetrics:')
    print(f'  Loss: {val_loss:.4f}')
    print(f'  Accuracy: {val_acc:.4f}')
    print(f'  Precision: {val_precision:.4f}')
    print(f'  Recall: {val_recall:.4f}')
    print(f'  F1-Score: {val_f1:.4f}')
    print(f'\nConfusion Matrix:')
    print(f'  True Positives: {val_tp}')
    print(f'  False Positives: {val_fp}')
    print(f'  True Negatives: {val_tn}')
    print(f'  False Negatives: {val_fn}')
else:
    # Regression metrics
    val_mae = np.abs(val_preds - val_targets).mean()
    val_rmse = np.sqrt(((val_preds - val_targets) ** 2).mean())
    val_r2 = 1 - (np.sum((val_targets - val_preds) ** 2) / np.sum((val_targets - val_targets.mean()) ** 2))
    
    # Calculate loss
    criterion = nn.MSELoss()
    val_loss = criterion(torch.FloatTensor(val_preds), torch.FloatTensor(val_targets)).item()
    
    print(f'\nMetrics:')
    print(f'  Loss (MSE): {val_loss:.4f}')
    print(f'  MAE: {val_mae:.4f}')
    print(f'  RMSE: {val_rmse:.4f}')
    print(f'  R² Score: {val_r2:.4f}')

print(f'\n{"="*60}')
print('\n💡 Use these metrics to tune hyperparameters.')
print('   Only evaluate on test set after you are done tuning!')

## 6. Save Model

Save the trained model for later evaluation.

In [ ]:
# Save final model with validation metrics in experiment directory
model_path = CONFIG['experiment_dir'] / 'final_model.pth'

if CONFIG['task_type'] == 'binary':
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': CONFIG,
        'input_shape': (X_train.shape[1], X_train.shape[2]),
        'task_type': 'binary',
        'val_accuracy': val_acc,
        'val_precision': val_precision,
        'val_recall': val_recall,
        'val_f1': val_f1,
        'val_loss': val_loss,
        'training_history': history
    }, model_path)

    # Save validation results
    val_results = {
        'task_type': 'binary',
        'val_loss': float(val_loss),
        'val_accuracy': float(val_acc),
        'val_precision': float(val_precision),
        'val_recall': float(val_recall),
        'val_f1': float(val_f1),
        'confusion_matrix': {
            'true_positives': int(val_tp),
            'false_positives': int(val_fp),
            'true_negatives': int(val_tn),
            'false_negatives': int(val_fn)
        }
    }
else:
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': CONFIG,
        'input_shape': (X_train.shape[1], X_train.shape[2]),
        'task_type': 'regression',
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'val_r2': val_r2,
        'val_loss': val_loss,
        'training_history': history
    }, model_path)

    # Save validation results
    val_results = {
        'task_type': 'regression',
        'val_loss': float(val_loss),
        'val_mae': float(val_mae),
        'val_rmse': float(val_rmse),
        'val_r2': float(val_r2)
    }

print(f'Model saved to: {model_path}')

# Save validation results in experiment directory
val_results_path = CONFIG['experiment_dir'] / 'val_results.json'
with open(val_results_path, 'w') as f:
    json.dump(val_results, f, indent=2)

print(f'Validation results saved to: {val_results_path}')

# Save configuration in experiment directory
config_path = CONFIG['experiment_dir'] / 'config.json'
with open(config_path, 'w') as f:
    json.dump({k: str(v) for k, v in CONFIG.items()}, f, indent=2)

print(f'Configuration saved to: {config_path}')

# Save training history plot
plt.savefig(CONFIG['experiment_dir'] / 'training_history.png', dpi=150, bbox_inches='tight')
print(f'Training plot saved to: {CONFIG["experiment_dir"] / "training_history.png"}')

# Track experiment in CSV
experiments_csv = CONFIG['save_dir'] / 'experiments.csv'

# Define all possible columns upfront
all_columns = [
    'experiment_name', 'timestamp', 'task_type', 'n_filters', 'dropout_rate',
    'learning_rate', 'batch_size', 'epochs_trained', 'best_epoch', 'notes',
    # Binary metrics
    'val_f1', 'val_accuracy', 'val_precision', 'val_recall',
    # Regression metrics
    'val_mae', 'val_rmse', 'val_r2',
    # Common
    'val_loss'
]

experiment_record = {
    'experiment_name': CONFIG['experiment_name'],
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'task_type': CONFIG['task_type'],
    'n_filters': CONFIG['n_filters'],
    'dropout_rate': CONFIG['dropout_rate'],
    'learning_rate': CONFIG['learning_rate'],
    'batch_size': CONFIG['batch_size'],
    'epochs_trained': history['total_epochs'],
    'best_epoch': history['best_epoch'],
    'notes': CONFIG.get('notes', ''),

}

# Add task-specific metrics, set others to None
if CONFIG['task_type'] == 'binary':
    experiment_record.update({
        'val_f1': float(val_f1),
        'val_accuracy': float(val_acc),
        'val_precision': float(val_precision),
        'val_recall': float(val_recall),
        'val_mae': None,
        'val_rmse': None,
        'val_r2': None,
        'val_loss': float(val_loss)
    })
else:
    experiment_record.update({
        'val_f1': None,
        'val_accuracy': None,
        'val_precision': None,
        'val_recall': None,
        'val_mae': float(val_mae),
        'val_rmse': float(val_rmse),
        'val_r2': float(val_r2),
        'val_loss': float(val_loss)
    })

# Append to CSV (create if doesn't exist)
import csv

file_exists = experiments_csv.exists()
with open(experiments_csv, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=all_columns)

    if not file_exists:
        writer.writeheader()
    writer.writerow(experiment_record)

print(f'Experiment tracked in: {experiments_csv}')

print('\n' + '=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'\nExperiment directory: {CONFIG["experiment_dir"]}')
print(f'\nNext step: Run final_evaluation.ipynb to evaluate on test set')
print(f'Model path: {model_path}')
print('\nTo compare experiments, check: {}'.format(experiments_csv))

# At the end of the cell:
CONFIG['model_type'] = MODEL_TYPE
